<div dir="rtl">

# الدرس 18: الانحدار اللوجستي: التصنيف بالاحتمالات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### بيانات من فئتين

</div>

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n = 100
X0 = rng.normal(loc=[-1, -1], scale=1.0, size=(n, 2))   # الفئة 0
X1 = rng.normal(loc=[1.5, 1.5], scale=1.0, size=(n, 2)) # الفئة 1
X = np.vstack([X0, X1])
y = np.array([0] * n + [1] * n)
print("X:", X.shape, " y:", y.shape, " classes:", np.bincount(y))

<div dir="rtl">

### من الصفر

</div>

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

class LogisticRegressionGD:
    def __init__(self, lr=0.1, epochs=2000):
        self.lr, self.epochs = lr, epochs

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.history = []
        for _ in range(self.epochs):
            p = self.predict_proba(X)
            err = p - y                               # الاحتمال ناقص الجواب
            self.w -= self.lr * X.T @ err / len(y)
            self.b -= self.lr * err.mean()
            eps = 1e-12
            self.history.append(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)))
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

clf = LogisticRegressionGD().fit(X, y)
print("w =", clf.w.round(3), " b =", round(clf.b, 3))
print("first loss:", round(clf.history[0], 3), " last loss:", round(clf.history[-1], 3))
print("training accuracy:", np.mean(clf.predict(X) == y))

<div dir="rtl">

### نرسم حد القرار

</div>

In [ ]:
import matplotlib.pyplot as plt

xx, yy = np.meshgrid(np.linspace(-4, 5, 200), np.linspace(-4, 5, 200))
grid = np.column_stack([xx.ravel(), yy.ravel()])
probs = clf.predict_proba(grid).reshape(xx.shape)

plt.figure(figsize=(5.4, 4.4))
cs = plt.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.35)
plt.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X0[:, 0], X0[:, 1], s=14, color="C0", label="class 0")
plt.scatter(X1[:, 0], X1[:, 1], s=14, color="C3", marker="s", label="class 1")
plt.colorbar(cs, label="p(class 1)")
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(loc="upper left")
plt.show()

<div dir="rtl">

### بمكتبة سايكت ليرن

</div>

In [ ]:
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression().fit(X, y)
print("sklearn w =", sk.coef_[0].round(3), " b =", sk.intercept_.round(3))
print("probabilities of first 3:", sk.predict_proba(X[:3]).round(3))
print("predictions of first 3  :", sk.predict(X[:3]))

<div dir="rtl">

### زهور السوسن: ثلاثة أنواع

</div>

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)

multi = LogisticRegression(max_iter=1000).fit(Xi_tr, yi_tr)
print("weights shape:", multi.coef_.shape)
print("test accuracy:", round(multi.score(Xi_te, yi_te), 3))
print("probabilities for one flower:", multi.predict_proba(Xi_te[:1]).round(3))

<div dir="rtl">

## تمرين: خطوة بيدك، ثم مرض حقيقي

الجزء الأول، على الورق:
مثالان: المدخل 1 والجواب 0، والمدخل 3 والجواب 1. ابدأ من وزن صفر وإزاحة صفر، ومعدل تعلّم 0.5.
احسب التدرّج، ثم الوزن والإزاحة بعد خطوة واحدة.

الجزء الثاني، بالكود:
مكتبة سايكت ليرن فيها بيانات أورام الثدي: 569 عينة، و 30 ميزة، وفئتان: خبيث أو حميد.
قسّم البيانات، ودرّب انحداراً لوجستياً داخل خط معالجة مع التوحيد القياسي، واطبع دقة الاختبار.

</div>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Xb, yb = load_breast_cancer(return_X_y=True)
# قسّم، ثم درّب، ثم قيّم